In [1]:
# ============================================================
# Section 1: Imports and Photos Library paths
# ============================================================

import osxphotos

from photos_inventory import *


USE_INVENTORY_CACHE = True


PHOTOS_LIBRARY_PATHS = {
    "backup_20250317": "/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary",
    "snapshot_20260603": "/Volumes/PRO-G40--20260519/Photos Library-Snapshot--20260603181919/Photos Library.photoslibrary",
    "test": "/Users/huohsien/Pictures/test.photoslibrary"
}


In [2]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

if USE_INVENTORY_CACHE:
    print("=" * 80)
    print("Load inventory cache: backup_20250317")
    print("=" * 80)

    inventory_backup = load_inventory_cache("backup_20250317")

    print()
    print("=" * 80)
    print("Load inventory cache: snapshot_20260603")
    print("=" * 80)

    inventory_snapshot = load_inventory_cache("snapshot_20260603")

else:
    print("=" * 80)
    print("Build inventory: backup_20250317")
    print("=" * 80)

    osx_assets = osxphotos.PhotosDB(
        PHOTOS_LIBRARY_PATHS["backup_20250317"]
    ).photos()

    print("backup osx asset count:", len(osx_assets))

    inventory_backup = build_inventory(osx_assets)

    print()
    print("Backup inventory summary")
    print("------------------------")
    print_inventory_summary(inventory_backup)

    save_inventory_cache(inventory_backup, "backup_20250317")

    print()
    print("=" * 80)
    print("Build inventory: snapshot_20260603")
    print("=" * 80)

    osx_assets_snapshot = osxphotos.PhotosDB(
        PHOTOS_LIBRARY_PATHS["snapshot_20260603"]
    ).photos()

    print("snapshot osx asset count:", len(osx_assets_snapshot))

    inventory_snapshot = build_inventory(osx_assets_snapshot)

    print()
    print("Snapshot inventory summary")
    print("--------------------------")
    print_inventory_summary(inventory_snapshot)

    save_inventory_cache(inventory_snapshot, "snapshot_20260603")


Load inventory cache: backup_20250317
loaded inventory cache: data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seconds: 0.24
inventory assets: 71607
inventory albums: 5172
inventory folders: 35
movies: 6240
hidden: 0
favorites: 699
descriptions: 728
keywords: 23758

Load inventory cache: snapshot_20260603
loaded inventory cache: data/inventory_cache/snapshot_20260603.inventory.pkl.gz
elapsed seconds: 0.38
inventory assets: 94202
inventory albums: 5942
inventory folders: 32
movies: 7366
hidden: 0
favorites: 763
descriptions: 2104
keywords: 27528


In [3]:
# ============================================================
# Test 2: Inventory comparison helpers
# ============================================================

from enum import Enum


class ChangeType(str, Enum):
    # Asset existence
    ASSET_MISSING_FROM_SNAPSHOT = "ASSET_MISSING_FROM_SNAPSHOT"
    ASSET_NEW_IN_SNAPSHOT = "ASSET_NEW_IN_SNAPSHOT"

    # Asset metadata fields
    ASSET_FIELD_CHANGED_DESCRIPTION = "ASSET_FIELD_CHANGED__description"
    ASSET_FIELD_CHANGED_KEYWORDS = "ASSET_FIELD_CHANGED__keywords"
    ASSET_FIELD_CHANGED_FAVORITE = "ASSET_FIELD_CHANGED__favorite"
    ASSET_FIELD_CHANGED_HIDDEN = "ASSET_FIELD_CHANGED__hidden"
    ASSET_FIELD_CHANGED_DATE = "ASSET_FIELD_CHANGED__date"
    ASSET_FIELD_CHANGED_DATE_ADDED = "ASSET_FIELD_CHANGED__date_added"
    ASSET_FIELD_CHANGED_ORIGINAL_FILENAME = "ASSET_FIELD_CHANGED__original_filename"
    ASSET_FIELD_CHANGED_IS_MOVIE = "ASSET_FIELD_CHANGED__is_movie"

    # Asset relationship metadata
    ASSET_ALBUM_MEMBERSHIP_REMOVED = "ASSET_ALBUM_MEMBERSHIP_REMOVED"
    ASSET_ALBUM_MEMBERSHIP_ADDED = "ASSET_ALBUM_MEMBERSHIP_ADDED"

    ASSET_FOLDER_PATHS_REMOVED = "ASSET_FOLDER_PATHS_REMOVED"
    ASSET_FOLDER_PATHS_ADDED = "ASSET_FOLDER_PATHS_ADDED"
    ASSET_FOLDER_PATHS_CHANGED = "ASSET_FOLDER_PATHS_CHANGED"

    # Album existence and folder relationship
    ALBUM_MISSING_FROM_SNAPSHOT = "ALBUM_MISSING_FROM_SNAPSHOT"
    ALBUM_NEW_IN_SNAPSHOT = "ALBUM_NEW_IN_SNAPSHOT"

    ALBUM_FOLDER_PATHS_REMOVED = "ALBUM_FOLDER_PATHS_REMOVED"
    ALBUM_FOLDER_PATHS_ADDED = "ALBUM_FOLDER_PATHS_ADDED"
    ALBUM_FOLDER_PATHS_CHANGED = "ALBUM_FOLDER_PATHS_CHANGED"

    ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS = "ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS"

    # Folder path existence
    FOLDER_PATH_MISSING_FROM_SNAPSHOT = "FOLDER_PATH_MISSING_FROM_SNAPSHOT"
    FOLDER_PATH_NEW_IN_SNAPSHOT = "FOLDER_PATH_NEW_IN_SNAPSHOT"


CHANGE_TYPE_DESCRIPTIONS = {
    ChangeType.ASSET_MISSING_FROM_SNAPSHOT:
        "Asset exists in backup but not in snapshot. This is high-priority because it may mean the photo or video itself disappeared after iCloud Photos crashes.",

    ChangeType.ASSET_NEW_IN_SNAPSHOT:
        "Asset exists in snapshot but not in backup. Usually normal because snapshot is later than backup.",

    ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION:
        "Same asset UUID exists in both libraries, but the caption/description changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_KEYWORDS:
        "Same asset UUID exists in both libraries, but keyword metadata changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_FAVORITE:
        "Same asset UUID exists in both libraries, but favorite status changed. This may be real user action or metadata loss.",

    ChangeType.ASSET_FIELD_CHANGED_HIDDEN:
        "Same asset UUID exists in both libraries, but hidden status changed. This may be real user action or metadata difference.",

    ChangeType.ASSET_FIELD_CHANGED_DATE:
        "Same asset UUID exists in both libraries, but asset date changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED:
        "Same asset UUID exists in both libraries, but date_added changed. This may be less important because import/sync timing can differ.",

    ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME:
        "Same asset UUID exists in both libraries, but original filename changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE:
        "Same asset UUID exists in both libraries, but is_movie changed. This is highly unusual.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED:
        "Asset still exists in snapshot, but one or more album memberships from backup are missing.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED:
        "Asset has album memberships in snapshot that did not exist in backup. Often normal because snapshot is later.",

    ChangeType.ASSET_FOLDER_PATHS_REMOVED:
        "Asset still exists in snapshot, but backup folder-path relationships are gone in snapshot.",

    ChangeType.ASSET_FOLDER_PATHS_ADDED:
        "Asset has folder-path relationships in snapshot that did not exist in backup. Often normal or caused by later organization.",

    ChangeType.ASSET_FOLDER_PATHS_CHANGED:
        "Asset still exists in both libraries, but folder-path relationships changed.",

    ChangeType.ALBUM_MISSING_FROM_SNAPSHOT:
        "Album title exists in backup but not in snapshot. This may require album reconstruction.",

    ChangeType.ALBUM_NEW_IN_SNAPSHOT:
        "Album title exists in snapshot but not in backup. Usually normal because snapshot is later.",

    ChangeType.ALBUM_FOLDER_PATHS_REMOVED:
        "Album still exists in snapshot, but it is no longer inside the folder path recorded in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_ADDED:
        "Album gained folder-path relationships in snapshot that did not exist in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_CHANGED:
        "Album still exists in both libraries, but its folder path changed.",

    ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS:
        "Album title is duplicated or ambiguous in at least one inventory, so title-based comparison is not reliable for this album.",

    ChangeType.FOLDER_PATH_MISSING_FROM_SNAPSHOT:
        "Folder path exists in backup but not in snapshot. Albums or assets may still exist elsewhere.",

    ChangeType.FOLDER_PATH_NEW_IN_SNAPSHOT:
        "Folder path exists in snapshot but not in backup. Usually normal if the folder was created later.",
}


FIELD_CHANGE_TYPES = {
    "description": ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION,
    "keywords": ChangeType.ASSET_FIELD_CHANGED_KEYWORDS,
    "favorite": ChangeType.ASSET_FIELD_CHANGED_FAVORITE,
    "hidden": ChangeType.ASSET_FIELD_CHANGED_HIDDEN,
    "date": ChangeType.ASSET_FIELD_CHANGED_DATE,
    "date_added": ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED,
    "original_filename": ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME,
    "is_movie": ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE,
}

## uuid among different photos libraries is different for the same asset

def make_asset_index(inventory):
    # Build photo_library_asset_unique_id -> Asset object.
    #
    # Photos UUID is local to one Photos Library database.
    # It cannot be used to match the same asset across different
    # Photos Library snapshots or backups.
    index = {}

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            raise RuntimeError(
                "Asset is missing photo_library_asset_unique_id. "
                "Run fill_photo_library_asset_unique_ids() first."
            )

        if unique_id in index:
            raise RuntimeError(
                "Duplicate photo_library_asset_unique_id found. "
                "Do not run comparison until the unique ID scheme is strengthened."
            )

        index[unique_id] = asset

    return index


def make_album_title_index(inventory):
    # Build album_title -> list[Album object].
    # Album title may not be globally unique, so keep a list.
    index = {}

    for album in inventory["albums"].values():
        title = album["title"] or ""

        if title not in index:
            index[title] = []

        index[title].append(album)

    return index


def make_folder_path_index(inventory):
    # Build folder_path -> list[Folder object].
    # Folder path may theoretically collide, so keep a list.
    index = {}

    for folder in inventory["folders"].values():
        path = folder["path"] or ""

        if path not in index:
            index[path] = []

        index[path].append(folder)

    return index


def asset_display_name(asset):
    # Prefer original filename for human reading.
    if asset is None:
        return None

    return asset["original_filename"] or asset["filename"] or asset["uuid"]


def album_folder_paths(album):
    # Return sorted folder paths for one Album object.
    return sorted(
        folder["path"]
        for folder in album["folders"].values()
    )


def asset_album_titles(asset):
    # Return sorted album titles for one Asset object.
    return sorted(
        album["title"] or ""
        for album in asset["albums"].values()
    )


def asset_folder_paths(asset):
    # Return sorted folder paths for one Asset object.
    return sorted(
        folder["path"] or ""
        for folder in asset["folders"].values()
    )


def add_diff(
    diff_records,
    change_type,
    scope,
    backup_object,
    snapshot_object,
    backup_value,
    snapshot_value,
    changed_field=None,
    note=None,
):
    # Add one normalized comparison record.
    if isinstance(change_type, ChangeType):
        change_type_value = change_type.value
        change_type_description = CHANGE_TYPE_DESCRIPTIONS.get(change_type)
    else:
        raise TypeError(f"change_type must be ChangeType, got: {change_type}")

    record = {
        "change_type": change_type_value,
        "change_type_description": change_type_description,
        "scope": scope,

        "changed_field": changed_field,

        "backup_object": backup_object,
        "snapshot_object": snapshot_object,

        "backup_value": backup_value,
        "snapshot_value": snapshot_value,

        "note": note,
    }

    diff_records.append(record)


def compare_asset_existence(inventory_backup, inventory_snapshot, diff_records):
    # Compare asset UUID existence.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    backup_uuids = set(backup_assets)
    snapshot_uuids = set(snapshot_assets)

    for asset_uuid in sorted(backup_uuids - snapshot_uuids):
        backup_asset = backup_assets[asset_uuid]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_MISSING_FROM_SNAPSHOT,
            scope="asset",
            backup_object=backup_asset,
            snapshot_object=None,
            backup_value=asset_display_name(backup_asset),
            snapshot_value=None,
            note="Asset exists in backup but not in snapshot. This is a high-priority possible iCloud crash data-loss case.",
        )

    for asset_uuid in sorted(snapshot_uuids - backup_uuids):
        snapshot_asset = snapshot_assets[asset_uuid]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_NEW_IN_SNAPSHOT,
            scope="asset",
            backup_object=None,
            snapshot_object=snapshot_asset,
            backup_value=None,
            snapshot_value=asset_display_name(snapshot_asset),
            note="Asset exists in snapshot but not in backup. This is usually normal because the snapshot is later.",
        )


def compare_asset_metadata(inventory_backup, inventory_snapshot, diff_records):
    # Compare metadata for assets with the same UUID.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    common_uuids = sorted(set(backup_assets) & set(snapshot_assets))

    fields_to_compare = [
        "original_filename",
        "is_movie",
        "date",
        "date_added",
        "description",
        "keywords",
        "favorite",
        "hidden",
    ]

    for asset_uuid in common_uuids:
        backup_asset = backup_assets[asset_uuid]
        snapshot_asset = snapshot_assets[asset_uuid]

        for field in fields_to_compare:
            backup_value = backup_asset.get(field)
            snapshot_value = snapshot_asset.get(field)

            if backup_value != snapshot_value:
                add_diff(
                    diff_records=diff_records,
                    change_type=FIELD_CHANGE_TYPES[field],
                    scope="asset",
                    backup_object=backup_asset,
                    snapshot_object=snapshot_asset,
                    backup_value=backup_value,
                    snapshot_value=snapshot_value,
                    changed_field=field,
                    note=f"Same asset UUID but asset field changed: {field}",
                )


def compare_asset_album_membership(inventory_backup, inventory_snapshot, diff_records):
    # Compare album membership by asset UUID and album title.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    common_uuids = sorted(set(backup_assets) & set(snapshot_assets))

    for asset_uuid in common_uuids:
        backup_asset = backup_assets[asset_uuid]
        snapshot_asset = snapshot_assets[asset_uuid]

        backup_titles = set(asset_album_titles(backup_asset))
        snapshot_titles = set(asset_album_titles(snapshot_asset))

        removed_titles = sorted(backup_titles - snapshot_titles)
        added_titles = sorted(snapshot_titles - backup_titles)

        if removed_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                snapshot_object=snapshot_asset,
                backup_value=removed_titles,
                snapshot_value=None,
                note="Asset still exists, but some backup album memberships are missing from snapshot.",
            )

        if added_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                snapshot_object=snapshot_asset,
                backup_value=None,
                snapshot_value=added_titles,
                note="Asset has album memberships in snapshot that did not exist in backup. Often normal for later snapshot.",
            )


def compare_asset_folder_paths(inventory_backup, inventory_snapshot, diff_records):
    # Compare folder paths attached to the same asset.
    # These are derived through asset -> album_info -> folder_list.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    common_uuids = sorted(set(backup_assets) & set(snapshot_assets))

    for asset_uuid in common_uuids:
        backup_asset = backup_assets[asset_uuid]
        snapshot_asset = snapshot_assets[asset_uuid]

        backup_paths = set(asset_folder_paths(backup_asset))
        snapshot_paths = set(asset_folder_paths(snapshot_asset))

        if backup_paths == snapshot_paths:
            continue

        if backup_paths and not snapshot_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_REMOVED
            note = "Asset still exists, but its folder paths are gone in snapshot."
        elif not backup_paths and snapshot_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_ADDED
            note = "Asset has folder paths in snapshot but did not have them in backup."
        else:
            change_type = ChangeType.ASSET_FOLDER_PATHS_CHANGED
            note = "Asset still exists, but folder paths changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="asset_folder_paths",
            backup_object=backup_asset,
            snapshot_object=snapshot_asset,
            backup_value=sorted(backup_paths),
            snapshot_value=sorted(snapshot_paths),
            note=note,
        )


def compare_album_existence_and_folder_paths(inventory_backup, inventory_snapshot, diff_records):
    # Compare albums by album title.
    # Title is the human-facing identity; UUID may not be stable across libraries.
    backup_albums_by_title = make_album_title_index(inventory_backup)
    snapshot_albums_by_title = make_album_title_index(inventory_snapshot)

    backup_titles = set(backup_albums_by_title)
    snapshot_titles = set(snapshot_albums_by_title)

    for title in sorted(backup_titles - snapshot_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_MISSING_FROM_SNAPSHOT,
            scope="album",
            backup_object=backup_albums_by_title[title],
            snapshot_object=None,
            backup_value=title,
            snapshot_value=None,
            note="Album title exists in backup but not in snapshot. This may require album reconstruction.",
        )

    for title in sorted(snapshot_titles - backup_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_NEW_IN_SNAPSHOT,
            scope="album",
            backup_object=None,
            snapshot_object=snapshot_albums_by_title[title],
            backup_value=None,
            snapshot_value=title,
            note="Album title exists in snapshot but not in backup. Usually normal for later snapshot.",
        )

    for title in sorted(backup_titles & snapshot_titles):
        backup_albums = backup_albums_by_title[title]
        snapshot_albums = snapshot_albums_by_title[title]

        if len(backup_albums) != 1 or len(snapshot_albums) != 1:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS,
                scope="album",
                backup_object=backup_albums,
                snapshot_object=snapshot_albums,
                backup_value=len(backup_albums),
                snapshot_value=len(snapshot_albums),
                note="Album title is not unique in at least one inventory. Folder comparison by title is ambiguous.",
            )
            continue

        backup_album = backup_albums[0]
        snapshot_album = snapshot_albums[0]

        backup_paths = set(album_folder_paths(backup_album))
        snapshot_paths = set(album_folder_paths(snapshot_album))

        if backup_paths == snapshot_paths:
            continue

        if backup_paths and not snapshot_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_REMOVED
            note = "Album still exists, but it is no longer inside any folder path in snapshot."
        elif not backup_paths and snapshot_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_ADDED
            note = "Album gained folder paths in snapshot."
        else:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_CHANGED
            note = "Album still exists, but its folder path changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="album_folder_paths",
            backup_object=backup_album,
            snapshot_object=snapshot_album,
            backup_value=sorted(backup_paths),
            snapshot_value=sorted(snapshot_paths),
            note=note,
        )


def compare_folder_paths(inventory_backup, inventory_snapshot, diff_records):
    # Compare folder paths by human-readable path.
    backup_folders_by_path = make_folder_path_index(inventory_backup)
    snapshot_folders_by_path = make_folder_path_index(inventory_snapshot)

    backup_paths = set(backup_folders_by_path)
    snapshot_paths = set(snapshot_folders_by_path)

    for path in sorted(backup_paths - snapshot_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_MISSING_FROM_SNAPSHOT,
            scope="folder",
            backup_object=backup_folders_by_path[path],
            snapshot_object=None,
            backup_value=path,
            snapshot_value=None,
            note="Folder path exists in backup but not in snapshot. Albums/assets may still exist elsewhere.",
        )

    for path in sorted(snapshot_paths - backup_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_NEW_IN_SNAPSHOT,
            scope="folder",
            backup_object=None,
            snapshot_object=snapshot_folders_by_path[path],
            backup_value=None,
            snapshot_value=path,
            note="Folder path exists in snapshot but not in backup.",
        )


def compare_inventories(inventory_backup, inventory_snapshot):
    # Run all comparison passes and return normalized diff records.
    diff_records = []

    compare_asset_existence(inventory_backup, inventory_snapshot, diff_records)
    compare_asset_metadata(inventory_backup, inventory_snapshot, diff_records)
    compare_asset_album_membership(inventory_backup, inventory_snapshot, diff_records)
    compare_asset_folder_paths(inventory_backup, inventory_snapshot, diff_records)
    compare_album_existence_and_folder_paths(inventory_backup, inventory_snapshot, diff_records)
    compare_folder_paths(inventory_backup, inventory_snapshot, diff_records)

    return diff_records


def summarize_diff_records(diff_records):
    # Count diff records by change_type.
    summary = {}

    for record in diff_records:
        change_type = record["change_type"]

        if change_type not in summary:
            summary[change_type] = 0

        summary[change_type] += 1

    return dict(sorted(summary.items()))


In [4]:
# ============================================================
# Test 2: Photo Library asset unique ID preflight
# ============================================================

fill_photo_library_asset_unique_ids(inventory_backup)
fill_photo_library_asset_unique_ids(inventory_snapshot)

backup_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_backup,
    "BACKUP Photo Library asset unique ID audit",
)

print()

snapshot_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_snapshot,
    "SNAPSHOT Photo Library asset unique ID audit",
)

print()
print("backup_unique_id_ok:", backup_unique_id_ok)
print("snapshot_unique_id_ok:", snapshot_unique_id_ok)
print("ready_for_cross_library_comparison:", backup_unique_id_ok and snapshot_unique_id_ok)

BACKUP Photo Library asset unique ID audit
--------------------------------------------------------------------------------
total asset count: 71607
generated unique ID count: 71607
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

SNAPSHOT Photo Library asset unique ID audit
--------------------------------------------------------------------------------
total asset count: 94202
generated unique ID count: 94202
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
is Photo Library asset unique ID scheme unique: True

backup_unique_id_ok: True
snapshot_unique_id_ok: True
ready_for_cross_library_comparison: True


In [5]:
# ============================================================
# Test 2: Run inventory comparison summary
# ============================================================

if not (backup_unique_id_ok and snapshot_unique_id_ok):
    raise RuntimeError(
        "Photo Library asset unique ID preflight failed. "
        "Do not run cross-library comparison yet."
    )

diff_records = compare_inventories(
    inventory_backup=inventory_backup,
    inventory_snapshot=inventory_snapshot,
)

In [6]:

# ============================================================
# Test 2: Inspect diff records by change type
# ============================================================

target_change_type = ChangeType.ASSET_MISSING_FROM_SNAPSHOT.value

matched_records = [
    record
    for record in diff_records
    if record["change_type"] == target_change_type
]

print("target change type:", target_change_type)
print("matched record count:", len(matched_records))
print()

for record in matched_records[:20]:
    backup_asset = record["backup_object"]
    snapshot_asset = record["snapshot_object"]

    print("change_type:", record["change_type"])
    print("description:", record["change_type_description"])
    print("changed_field:", record["changed_field"])
    print("backup_value:", record["backup_value"])
    print("snapshot_value:", record["snapshot_value"])

    if backup_asset is not None:
        print("backup uuid:", backup_asset["uuid"])
        print("backup original_filename:", backup_asset["original_filename"])
        print("backup date:", backup_asset["date"])

    if snapshot_asset is not None:
        print("snapshot uuid:", snapshot_asset["uuid"])
        print("snapshot original_filename:", snapshot_asset["original_filename"])
        print("snapshot date:", snapshot_asset["date"])

    print("-" * 80)


target change type: ASSET_MISSING_FROM_SNAPSHOT
matched record count: 15

change_type: ASSET_MISSING_FROM_SNAPSHOT
description: Asset exists in backup but not in snapshot. This is high-priority because it may mean the photo or video itself disappeared after iCloud Photos crashes.
changed_field: None
backup_value: IMG_0003.PNG
snapshot_value: None
backup uuid: 7F2FBB82-3F67-41D7-8AA6-338515BFBFCE
backup original_filename: IMG_0003.PNG
backup date: 2025-03-24T01:35:51+08:00
--------------------------------------------------------------------------------
change_type: ASSET_MISSING_FROM_SNAPSHOT
description: Asset exists in backup but not in snapshot. This is high-priority because it may mean the photo or video itself disappeared after iCloud Photos crashes.
changed_field: None
backup_value: IMG_0024.JPG
snapshot_value: None
backup uuid: 9C7310F3-B2C3-465D-9D9D-2EBC46577D7C
backup original_filename: IMG_0024.JPG
backup date: 2018-05-21T11:19:16.402110+08:00
--------------------------------

In [7]:
# # ============================================================
# # Test 2: Diagnose asset identity matching
# # ============================================================

# backup_assets_by_uuid = make_asset_index(inventory_backup)
# snapshot_assets_by_uuid = make_asset_index(inventory_snapshot)

# backup_uuids = set(backup_assets_by_uuid)
# snapshot_uuids = set(snapshot_assets_by_uuid)

# common_uuids = backup_uuids & snapshot_uuids
# backup_only_uuids = backup_uuids - snapshot_uuids
# snapshot_only_uuids = snapshot_uuids - backup_uuids

# print("backup asset count:", len(backup_uuids))
# print("snapshot asset count:", len(snapshot_uuids))
# print("common uuid count:", len(common_uuids))
# print("backup-only uuid count:", len(backup_only_uuids))
# print("snapshot-only uuid count:", len(snapshot_only_uuids))
# print()

# if len(common_uuids) == 0:
#     print("WARNING: No shared asset UUIDs. UUID is not a usable cross-library identity key here.")

In [8]:
# # ============================================================
# # Test 2: Diagnose external asset identity key
# # ============================================================

# def make_asset_external_key(asset):
#     # Candidate cross-library identity key.
#     return (
#         asset["original_filename"],
#         asset["date"],
#         asset["is_movie"],
#     )


# def make_asset_external_key_index(inventory):
#     # key -> list[Asset object]
#     index = {}

#     for asset in inventory["assets"]:
#         key = make_asset_external_key(asset)

#         if key not in index:
#             index[key] = []

#         index[key].append(asset)

#     return index


# backup_assets_by_key = make_asset_external_key_index(inventory_backup)
# snapshot_assets_by_key = make_asset_external_key_index(inventory_snapshot)

# backup_keys = set(backup_assets_by_key)
# snapshot_keys = set(snapshot_assets_by_key)

# common_keys = backup_keys & snapshot_keys
# backup_only_keys = backup_keys - snapshot_keys
# snapshot_only_keys = snapshot_keys - backup_keys

# backup_duplicate_keys = {
#     key: assets
#     for key, assets in backup_assets_by_key.items()
#     if len(assets) > 1
# }

# snapshot_duplicate_keys = {
#     key: assets
#     for key, assets in snapshot_assets_by_key.items()
#     if len(assets) > 1
# }

# print("backup external key count:", len(backup_keys))
# print("snapshot external key count:", len(snapshot_keys))
# print("common external key count:", len(common_keys))
# print("backup-only external key count:", len(backup_only_keys))
# print("snapshot-only external key count:", len(snapshot_only_keys))
# print()

# print("backup duplicate external key count:", len(backup_duplicate_keys))
# print("snapshot duplicate external key count:", len(snapshot_duplicate_keys))

In [9]:
# # ============================================================
# # Test 2: Inspect backup-only external keys
# # ============================================================

# backup_only_unique_keys = [
#     key
#     for key in sorted(backup_only_keys)
#     if len(backup_assets_by_key[key]) == 1
# ]

# snapshot_only_unique_keys = [
#     key
#     for key in sorted(snapshot_only_keys)
#     if len(snapshot_assets_by_key[key]) == 1
# ]

# print("backup-only unique key count:", len(backup_only_unique_keys))
# print("snapshot-only unique key count:", len(snapshot_only_unique_keys))
# print()

# for key in backup_only_unique_keys[:50]:
#     asset = backup_assets_by_key[key][0]

#     print("external key:", key)
#     print("backup uuid:", asset["uuid"])
#     print("original_filename:", asset["original_filename"])
#     print("filename:", asset["filename"])
#     print("date:", asset["date"])
#     print("is_movie:", asset["is_movie"])
#     print("description:", asset["description"])
#     print("keywords:", asset["keywords"])
#     print("albums:", asset_album_titles(asset))
#     print("folders:", asset_folder_paths(asset))
#     print("-" * 80)

In [10]:
# # ============================================================
# # Test 2: Diagnose backup-only matching keys
# # ============================================================

# from datetime import datetime


# def parse_iso_datetime(value):
#     # Parse ISO datetime string safely.
#     if value is None:
#         return None

#     try:
#         return datetime.fromisoformat(value)
#     except ValueError:
#         return None


# def seconds_between(date_string_1, date_string_2):
#     # Return absolute seconds between two ISO datetime strings.
#     date_1 = parse_iso_datetime(date_string_1)
#     date_2 = parse_iso_datetime(date_string_2)

#     if date_1 is None or date_2 is None:
#         return None

#     return abs((date_1 - date_2).total_seconds())


# def make_snapshot_lookup_tables(inventory_snapshot):
#     # Build snapshot lookup tables for diagnosing unmatched backup assets.
#     by_original_filename = {}
#     by_date_and_is_movie = {}
#     by_filename_number_and_is_movie = {}

#     for asset in inventory_snapshot["assets"]:
#         original_filename = asset["original_filename"]
#         date = asset["date"]
#         is_movie = asset["is_movie"]

#         by_original_filename.setdefault(original_filename, []).append(asset)
#         by_date_and_is_movie.setdefault((date, is_movie), []).append(asset)

#         filename_number = extract_img_number(original_filename)
#         by_filename_number_and_is_movie.setdefault((filename_number, is_movie), []).append(asset)

#     return by_original_filename, by_date_and_is_movie, by_filename_number_and_is_movie


# def extract_img_number(original_filename):
#     # Extract numeric part from filenames like IMG_1234.JPG.
#     if original_filename is None:
#         return None

#     name = original_filename.upper()

#     if not name.startswith("IMG_"):
#         return None

#     number_part = name[4:].split(".")[0]

#     if not number_part.isdigit():
#         return None

#     return number_part


# def diagnose_backup_only_asset(asset, snapshot_lookup_tables):
#     # Diagnose whether one backup-only asset has possible snapshot matches.
#     (
#         snapshot_by_original_filename,
#         snapshot_by_date_and_is_movie,
#         snapshot_by_filename_number_and_is_movie,
#     ) = snapshot_lookup_tables

#     original_filename = asset["original_filename"]
#     date = asset["date"]
#     is_movie = asset["is_movie"]
#     filename_number = extract_img_number(original_filename)

#     same_original_filename_candidates = snapshot_by_original_filename.get(original_filename, [])
#     same_date_candidates = snapshot_by_date_and_is_movie.get((date, is_movie), [])
#     same_filename_number_candidates = snapshot_by_filename_number_and_is_movie.get((filename_number, is_movie), [])

#     near_time_candidates = []

#     for candidate in same_original_filename_candidates:
#         seconds = seconds_between(date, candidate["date"])

#         if seconds is not None and seconds <= 5:
#             near_time_candidates.append((seconds, candidate))

#     diagnosis = {
#         "backup_asset": asset,
#         "same_original_filename_count": len(same_original_filename_candidates),
#         "same_date_and_is_movie_count": len(same_date_candidates),
#         "same_filename_number_and_is_movie_count": len(same_filename_number_candidates),
#         "near_time_same_original_filename_count": len(near_time_candidates),
#         "same_original_filename_candidates": same_original_filename_candidates,
#         "same_date_candidates": same_date_candidates,
#         "same_filename_number_candidates": same_filename_number_candidates,
#         "near_time_candidates": near_time_candidates,
#     }

#     return diagnosis


# snapshot_lookup_tables = make_snapshot_lookup_tables(inventory_snapshot)

# backup_only_diagnostics = []

# for key in backup_only_unique_keys:
#     asset = backup_assets_by_key[key][0]
#     diagnosis = diagnose_backup_only_asset(asset, snapshot_lookup_tables)
#     backup_only_diagnostics.append(diagnosis)


# summary_counts = {
#     "no_candidate": 0,
#     "same_original_filename_exists": 0,
#     "same_date_and_is_movie_exists": 0,
#     "same_filename_number_and_is_movie_exists": 0,
#     "near_time_same_original_filename_exists": 0,
# }

# for diagnosis in backup_only_diagnostics:
#     has_any_candidate = False

#     if diagnosis["same_original_filename_count"] > 0:
#         summary_counts["same_original_filename_exists"] += 1
#         has_any_candidate = True

#     if diagnosis["same_date_and_is_movie_count"] > 0:
#         summary_counts["same_date_and_is_movie_exists"] += 1
#         has_any_candidate = True

#     if diagnosis["same_filename_number_and_is_movie_count"] > 0:
#         summary_counts["same_filename_number_and_is_movie_exists"] += 1
#         has_any_candidate = True

#     if diagnosis["near_time_same_original_filename_count"] > 0:
#         summary_counts["near_time_same_original_filename_exists"] += 1
#         has_any_candidate = True

#     if not has_any_candidate:
#         summary_counts["no_candidate"] += 1


# print("backup-only unique key count:", len(backup_only_unique_keys))
# print()

# for name, count in summary_counts.items():
#     print(name, ":", count)

In [11]:
# # ============================================================
# # Test 2: Print backup-only matching key diagnostics
# # ============================================================

# for diagnosis in backup_only_diagnostics[:50]:
#     asset = diagnosis["backup_asset"]

#     print("backup original_filename:", asset["original_filename"])
#     print("backup filename:", asset["filename"])
#     print("backup uuid:", asset["uuid"])
#     print("backup date:", asset["date"])
#     print("backup is_movie:", asset["is_movie"])
#     print("backup description:", asset["description"])
#     print("backup keywords:", asset["keywords"])
#     print("backup albums:", asset_album_titles(asset))
#     print("backup folders:", asset_folder_paths(asset))
#     print()

#     print("same_original_filename_count:", diagnosis["same_original_filename_count"])
#     print("same_date_and_is_movie_count:", diagnosis["same_date_and_is_movie_count"])
#     print("same_filename_number_and_is_movie_count:", diagnosis["same_filename_number_and_is_movie_count"])
#     print("near_time_same_original_filename_count:", diagnosis["near_time_same_original_filename_count"])
#     print()

#     if diagnosis["same_original_filename_candidates"]:
#         print("Same original_filename candidates")
#         for candidate in diagnosis["same_original_filename_candidates"][:5]:
#             print("  snapshot original_filename:", candidate["original_filename"])
#             print("  snapshot uuid:", candidate["uuid"])
#             print("  snapshot date:", candidate["date"])
#             print("  snapshot is_movie:", candidate["is_movie"])
#             print("  snapshot albums:", asset_album_titles(candidate))
#             print("  snapshot folders:", asset_folder_paths(candidate))
#             print()

#     if diagnosis["same_date_candidates"]:
#         print("Same date + is_movie candidates")
#         for candidate in diagnosis["same_date_candidates"][:5]:
#             print("  snapshot original_filename:", candidate["original_filename"])
#             print("  snapshot uuid:", candidate["uuid"])
#             print("  snapshot date:", candidate["date"])
#             print("  snapshot is_movie:", candidate["is_movie"])
#             print()

#     print("-" * 80)

In [12]:
# # ============================================================
# # Test 2: Classify backup-only matching key failures
# # ============================================================

# from datetime import datetime, timezone


# def normalize_filename_for_matching(original_filename):
#     # Normalize filename for cross-library matching.
#     # Example: IMG_0003.PNG and IMG_0003.jpg both become IMG_0003.
#     if original_filename is None:
#         return None

#     name = original_filename.strip().upper()

#     if "." in name:
#         name = ".".join(name.split(".")[:-1])

#     return name


# def datetime_to_utc_timestamp(date_string):
#     # Convert ISO datetime string to UTC timestamp.
#     if date_string is None:
#         return None

#     try:
#         dt = datetime.fromisoformat(date_string)
#     except ValueError:
#         return None

#     if dt.tzinfo is None:
#         return None

#     return round(dt.timestamp(), 6)


# def make_stronger_asset_matching_key(asset):
#     # Stronger candidate key:
#     # same base filename + same absolute time + same media type.
#     return (
#         normalize_filename_for_matching(asset["original_filename"]),
#         datetime_to_utc_timestamp(asset["date"]),
#         asset["is_movie"],
#     )


# def make_stronger_asset_matching_key_index(inventory):
#     # key -> list[Asset object]
#     index = {}

#     for asset in inventory["assets"]:
#         key = make_stronger_asset_matching_key(asset)

#         if key not in index:
#             index[key] = []

#         index[key].append(asset)

#     return index


# backup_assets_by_stronger_key = make_stronger_asset_matching_key_index(inventory_backup)
# snapshot_assets_by_stronger_key = make_stronger_asset_matching_key_index(inventory_snapshot)

# backup_stronger_keys = set(backup_assets_by_stronger_key)
# snapshot_stronger_keys = set(snapshot_assets_by_stronger_key)

# common_stronger_keys = backup_stronger_keys & snapshot_stronger_keys
# backup_only_stronger_keys = backup_stronger_keys - snapshot_stronger_keys
# snapshot_only_stronger_keys = snapshot_stronger_keys - backup_stronger_keys

# backup_duplicate_stronger_keys = {
#     key: assets
#     for key, assets in backup_assets_by_stronger_key.items()
#     if len(assets) > 1
# }

# snapshot_duplicate_stronger_keys = {
#     key: assets
#     for key, assets in snapshot_assets_by_stronger_key.items()
#     if len(assets) > 1
# }

# print("backup stronger key count:", len(backup_stronger_keys))
# print("snapshot stronger key count:", len(snapshot_stronger_keys))
# print("common stronger key count:", len(common_stronger_keys))
# print("backup-only stronger key count:", len(backup_only_stronger_keys))
# print("snapshot-only stronger key count:", len(snapshot_only_stronger_keys))
# print()

# print("backup duplicate stronger key count:", len(backup_duplicate_stronger_keys))
# print("snapshot duplicate stronger key count:", len(snapshot_duplicate_stronger_keys))

In [13]:
# # ============================================================
# # Test 2: Inspect backup-only stronger keys
# # ============================================================

# print("backup-only stronger key count:", len(backup_only_stronger_keys))
# print()

# for key in sorted(backup_only_stronger_keys):
#     assets = backup_assets_by_stronger_key[key]

#     print("stronger key:", key)
#     print("asset count:", len(assets))

#     for asset in assets:
#         print("backup uuid:", asset["uuid"])
#         print("original_filename:", asset["original_filename"])
#         print("filename:", asset["filename"])
#         print("date:", asset["date"])
#         print("is_movie:", asset["is_movie"])
#         print("description:", asset["description"])
#         print("keywords:", asset["keywords"])
#         print("albums:", asset_album_titles(asset))
#         print("folders:", asset_folder_paths(asset))
#         print()

#     print("-" * 80)

In [14]:
# # ============================================================
# # Test 2: Diagnose whether backup-only stronger keys are truly missing
# # ============================================================

# def same_month_day_time(date_string_1, date_string_2):
#     # True if month/day/hour/minute/second are the same, ignoring year.
#     dt1 = parse_iso_datetime(date_string_1)
#     dt2 = parse_iso_datetime(date_string_2)

#     if dt1 is None or dt2 is None:
#         return False

#     return (
#         dt1.month == dt2.month
#         and dt1.day == dt2.day
#         and dt1.hour == dt2.hour
#         and dt1.minute == dt2.minute
#         and dt1.second == dt2.second
#     )


# def diagnose_possible_missing_asset(backup_asset, inventory_snapshot):
#     # Diagnose whether a backup-only stronger-key asset is truly missing.
#     backup_base_name = normalize_filename_for_matching(
#         backup_asset["original_filename"]
#     )

#     backup_timestamp = datetime_to_utc_timestamp(
#         backup_asset["date"]
#     )

#     same_base_name_candidates = []
#     same_month_day_time_candidates = []
#     same_timestamp_candidates = []

#     for snapshot_asset in inventory_snapshot["assets"]:
#         snapshot_base_name = normalize_filename_for_matching(
#             snapshot_asset["original_filename"]
#         )

#         snapshot_timestamp = datetime_to_utc_timestamp(
#             snapshot_asset["date"]
#         )

#         if snapshot_base_name == backup_base_name:
#             same_base_name_candidates.append(snapshot_asset)

#             if same_month_day_time(
#                 backup_asset["date"],
#                 snapshot_asset["date"],
#             ):
#                 same_month_day_time_candidates.append(snapshot_asset)

#         if (
#             backup_timestamp is not None
#             and snapshot_timestamp is not None
#             and backup_timestamp == snapshot_timestamp
#         ):
#             same_timestamp_candidates.append(snapshot_asset)

#     if same_month_day_time_candidates:
#         diagnosis = "POSSIBLE_YEAR_CHANGED"
#     elif same_timestamp_candidates:
#         diagnosis = "POSSIBLE_FILENAME_CHANGED"
#     elif same_base_name_candidates:
#         diagnosis = "POSSIBLE_DATE_CHANGED"
#     else:
#         diagnosis = "LIKELY_FILE_MISSING"

#     return {
#         "backup_asset": backup_asset,
#         "diagnosis": diagnosis,
#         "same_base_name_candidates": same_base_name_candidates,
#         "same_month_day_time_candidates": same_month_day_time_candidates,
#         "same_timestamp_candidates": same_timestamp_candidates,
#     }


# missing_asset_diagnostics = []

# for key in sorted(backup_only_stronger_keys):
#     for backup_asset in backup_assets_by_stronger_key[key]:
#         diagnosis = diagnose_possible_missing_asset(
#             backup_asset=backup_asset,
#             inventory_snapshot=inventory_snapshot,
#         )

#         missing_asset_diagnostics.append(diagnosis)


# diagnosis_summary = {}

# for item in missing_asset_diagnostics:
#     diagnosis = item["diagnosis"]

#     if diagnosis not in diagnosis_summary:
#         diagnosis_summary[diagnosis] = 0

#     diagnosis_summary[diagnosis] += 1


# print("backup-only stronger key asset count:", len(missing_asset_diagnostics))
# print()

# for diagnosis, count in sorted(diagnosis_summary.items()):
#     print(diagnosis, ":", count)

In [15]:
# # ============================================================
# # Test 2: Print possible missing asset diagnosis details
# # ============================================================

# for item in missing_asset_diagnostics:
#     backup_asset = item["backup_asset"]

#     print("=" * 100)
#     print("DIAGNOSIS:", item["diagnosis"])
#     print()
#     print("BACKUP")
#     print("uuid:", backup_asset["uuid"])
#     print("original_filename:", backup_asset["original_filename"])
#     print("filename:", backup_asset["filename"])
#     print("date:", backup_asset["date"])
#     print("is_movie:", backup_asset["is_movie"])
#     print("description:", backup_asset["description"])
#     print("keywords:", backup_asset["keywords"])
#     print("albums:", asset_album_titles(backup_asset))
#     print("folders:", asset_folder_paths(backup_asset))
#     print()

#     print("same_base_name_candidates:", len(item["same_base_name_candidates"]))
#     print("same_month_day_time_candidates:", len(item["same_month_day_time_candidates"]))
#     print("same_timestamp_candidates:", len(item["same_timestamp_candidates"]))
#     print()

#     if item["same_month_day_time_candidates"]:
#         print("SAME MONTH/DAY/TIME CANDIDATES")
#         for candidate in item["same_month_day_time_candidates"][:10]:
#             print("  snapshot uuid:", candidate["uuid"])
#             print("  snapshot original_filename:", candidate["original_filename"])
#             print("  snapshot date:", candidate["date"])
#             print("  snapshot keywords:", candidate["keywords"])
#             print("  snapshot albums:", asset_album_titles(candidate))
#             print("  snapshot folders:", asset_folder_paths(candidate))
#             print()

#     elif item["same_timestamp_candidates"]:
#         print("SAME TIMESTAMP CANDIDATES")
#         for candidate in item["same_timestamp_candidates"][:10]:
#             print("  snapshot uuid:", candidate["uuid"])
#             print("  snapshot original_filename:", candidate["original_filename"])
#             print("  snapshot date:", candidate["date"])
#             print("  snapshot keywords:", candidate["keywords"])
#             print("  snapshot albums:", asset_album_titles(candidate))
#             print("  snapshot folders:", asset_folder_paths(candidate))
#             print()

#     elif item["same_base_name_candidates"]:
#         print("SAME BASE NAME CANDIDATES")
#         for candidate in item["same_base_name_candidates"][:10]:
#             print("  snapshot uuid:", candidate["uuid"])
#             print("  snapshot original_filename:", candidate["original_filename"])
#             print("  snapshot date:", candidate["date"])
#             print("  snapshot keywords:", candidate["keywords"])
#             print("  snapshot albums:", asset_album_titles(candidate))
#             print("  snapshot folders:", asset_folder_paths(candidate))
#             print()

#     print("-" * 100)

In [16]:
# # ============================================================
# # Test 2: File-content check for backup-only stronger-key assets
# # ============================================================

# import hashlib
# import os


# def sha256_file(path, chunk_size=1024 * 1024):
#     # Compute SHA256 for one file path.
#     if path is None:
#         return None

#     if not os.path.exists(path):
#         return None

#     hasher = hashlib.sha256()

#     with open(path, "rb") as f:
#         while True:
#             chunk = f.read(chunk_size)

#             if not chunk:
#                 break

#             hasher.update(chunk)

#     return hasher.hexdigest()


# def file_size(path):
#     # Return file size in bytes.
#     if path is None:
#         return None

#     if not os.path.exists(path):
#         return None

#     return os.path.getsize(path)


# backup_only_stronger_assets = []

# for key in sorted(backup_only_stronger_keys):
#     for asset in backup_assets_by_stronger_key[key]:
#         backup_only_stronger_assets.append(asset)

# print("backup-only stronger asset count:", len(backup_only_stronger_assets))

In [17]:
# # ============================================================
# # Test 2: Build snapshot file-size index
# # ============================================================

# snapshot_assets_by_size = {}

# for asset in inventory_snapshot["assets"]:
#     path = asset["path"]
#     size = file_size(path)

#     if size is None:
#         continue

#     if size not in snapshot_assets_by_size:
#         snapshot_assets_by_size[size] = []

#     snapshot_assets_by_size[size].append(asset)

# print("snapshot file-size bucket count:", len(snapshot_assets_by_size))

In [18]:
# # ============================================================
# # Test 2: Check whether backup-only files exist in snapshot by SHA256
# # ============================================================

# file_content_results = []

# for backup_asset in backup_only_stronger_assets:
#     backup_path = backup_asset["path"]
#     backup_size = file_size(backup_path)
#     backup_sha256 = sha256_file(backup_path)

#     same_size_snapshot_assets = snapshot_assets_by_size.get(backup_size, [])

#     matched_snapshot_assets = []

#     for snapshot_asset in same_size_snapshot_assets:
#         snapshot_path = snapshot_asset["path"]
#         snapshot_sha256 = sha256_file(snapshot_path)

#         if snapshot_sha256 == backup_sha256:
#             matched_snapshot_assets.append(snapshot_asset)

#     if matched_snapshot_assets:
#         result_type = "FILE_CONTENT_FOUND_IN_SNAPSHOT"
#     else:
#         result_type = "FILE_CONTENT_NOT_FOUND_IN_SNAPSHOT"

#     file_content_results.append({
#         "result_type": result_type,
#         "backup_asset": backup_asset,
#         "backup_path": backup_path,
#         "backup_size": backup_size,
#         "backup_sha256": backup_sha256,
#         "same_size_snapshot_count": len(same_size_snapshot_assets),
#         "matched_snapshot_assets": matched_snapshot_assets,
#     })


# summary = {}

# for result in file_content_results:
#     result_type = result["result_type"]

#     if result_type not in summary:
#         summary[result_type] = 0

#     summary[result_type] += 1


# print("file content result summary")
# print("---------------------------")

# for result_type, count in sorted(summary.items()):
#     print(result_type, ":", count)

In [19]:
# # ============================================================
# # Test 2: Print file-content check details
# # ============================================================

# for result in file_content_results:
#     backup_asset = result["backup_asset"]

#     print("=" * 100)
#     print("RESULT:", result["result_type"])
#     print()

#     print("BACKUP")
#     print("uuid:", backup_asset["uuid"])
#     print("original_filename:", backup_asset["original_filename"])
#     print("date:", backup_asset["date"])
#     print("path:", result["backup_path"])
#     print("size:", result["backup_size"])
#     print("sha256:", result["backup_sha256"])
#     print("keywords:", backup_asset["keywords"])
#     print("albums:", asset_album_titles(backup_asset))
#     print("folders:", asset_folder_paths(backup_asset))
#     print()

#     print("same-size snapshot candidate count:", result["same_size_snapshot_count"])
#     print("matched snapshot asset count:", len(result["matched_snapshot_assets"]))
#     print()

#     for snapshot_asset in result["matched_snapshot_assets"]:
#         print("MATCHED SNAPSHOT")
#         print("uuid:", snapshot_asset["uuid"])
#         print("original_filename:", snapshot_asset["original_filename"])
#         print("date:", snapshot_asset["date"])
#         print("path:", snapshot_asset["path"])
#         print("keywords:", snapshot_asset["keywords"])
#         print("albums:", asset_album_titles(snapshot_asset))
#         print("folders:", asset_folder_paths(snapshot_asset))
#         print()

#     print("-" * 100)

In [20]:
# # ============================================================
# # Test 2: Inspect one asset base filename across backup and snapshot
# # ============================================================

# target_base_name = "IMG_0167"


# def get_asset_base_name(asset):
#     # Return normalized base filename without extension.
#     return normalize_filename_for_matching(asset["original_filename"])


# def find_assets_by_base_name(inventory, base_name):
#     # Find all assets whose normalized base filename matches base_name.
#     base_name = base_name.upper()

#     return [
#         asset
#         for asset in inventory["assets"]
#         if get_asset_base_name(asset) == base_name
#     ]


# def asset_has_album_keyword(asset, keyword):
#     # Check whether any album title contains a keyword.
#     return any(
#         keyword in title
#         for title in asset_album_titles(asset)
#     )


# def print_asset_debug(asset, label):
#     # Print one asset with matching/debug fields.
#     path = asset["path"]
#     size = file_size(path)
#     sha256 = sha256_file(path)

#     print(label)
#     print("uuid:", asset["uuid"])
#     print("original_filename:", asset["original_filename"])
#     print("filename:", asset["filename"])
#     print("date:", asset["date"])
#     print("is_movie:", asset["is_movie"])
#     print("stronger key:", make_stronger_asset_matching_key(asset))
#     print("path:", path)
#     print("size:", size)
#     print("sha256:", sha256)
#     print("keywords:", asset["keywords"])
#     print("albums:", asset_album_titles(asset))
#     print("folders:", asset_folder_paths(asset))
#     print()


# backup_target_assets = find_assets_by_base_name(
#     inventory_backup,
#     target_base_name,
# )

# snapshot_target_assets = find_assets_by_base_name(
#     inventory_snapshot,
#     target_base_name,
# )

# print("target base name:", target_base_name)
# print("backup target asset count:", len(backup_target_assets))
# print("snapshot target asset count:", len(snapshot_target_assets))
# print()

# print("=" * 100)
# print("BACKUP TARGET ASSETS")
# print("=" * 100)

# for backup_asset in backup_target_assets:
#     backup_key = make_stronger_asset_matching_key(backup_asset)
#     snapshot_matches_by_key = snapshot_assets_by_stronger_key.get(
#         backup_key,
#         [],
#     )

#     print_asset_debug(backup_asset, "BACKUP")

#     print("snapshot matches by stronger key:", len(snapshot_matches_by_key))

#     for snapshot_asset in snapshot_matches_by_key:
#         print_asset_debug(snapshot_asset, "MATCHED SNAPSHOT BY STRONGER KEY")

#     print("-" * 100)


# print()
# print("=" * 100)
# print("SNAPSHOT TARGET ASSETS CONTAINING 資產配置 OR 2024/2120 DATE AREA")
# print("=" * 100)

# for snapshot_asset in snapshot_target_assets:
#     date_string = snapshot_asset["date"] or ""

#     looks_relevant = (
#         asset_has_album_keyword(snapshot_asset, "資產配置")
#         or "2024-11-28T13:" in date_string
#         or "2120-11-28T13:" in date_string
#     )

#     if not looks_relevant:
#         continue

#     print_asset_debug(snapshot_asset, "SNAPSHOT RELEVANT CANDIDATE")
#     print("-" * 100)